# 03 — Run the Trained Model Live

Load the MOUSE model pushed by `02_train_offline.ipynb` and run it in the same Procedural Frozen Lake environment it was trained on.

At each step the model receives the most recent `(action, observation, reward, done)` and picks the action with the highest predicted score. We run several episodes and report the success rate (fraction of episodes that reach the goal), then capture one episode frame-by-frame and play it back as an inline animation.

In [1]:
import os

import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from mouse_envs import EnvConfig, make_env
from mouse_core import load_model

# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

# Hub model repo written by 02_train_offline.ipynb.
MODEL_ID = "mouse-example-model"

# FrozenLake has four discrete actions.
MAX_ACTIONS = 4

# Number of episodes for the success-rate evaluation.
EVAL_EPISODES = 20

# Number of steps to capture for the replay animation.
ANIMATION_STEPS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Environment

We use the same fixed 4×4 map and settings as the training notebook so the model operates in the distribution it was trained on.

In [ ]:
FIXED_MAP = ["SFFF", "FHFH", "FFFH", "HFFG"]

env = make_env(EnvConfig(
    "Procedural-FrozenLake-v1",
    name="proc_frozenlake",
    seed=0,
    num_envs=1,
    episodes_per_task=EVAL_EPISODES,
    kwargs={"is_slippery": False, "fixed_map": FIXED_MAP, "step_penalty": -0.05, "render_mode": "rgb_array"},
))

## Load model

Download the checkpoint from the Hub if needed, reconstruct the saved MOUSE model, and move it to the selected device.

In [3]:
model = load_model(MODEL_ID, map_location="cpu").eval().to(device)

pytorch_model.bin:   0%|          | 0.00/126M [00:00<?, ?B/s]

## Incremental inference with KV-cache

At inference time you don't need to re-process the full history on every step. Backbones that support it (`LlamaBackbone`, `Qwen3Backbone`) cache the key/value states from previous tokens so each new step only processes the single new token.

Pass `use_cache=True` and thread the cache object across steps:

```python
cache = None

with torch.no_grad():
    for step in range(EVAL_STEPS):
        predictions, _, cache = model(batch, cache=cache, use_cache=True)
        action = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
```

`model.get_action` reads the configured action head from the last step of `predictions`. Use `temperature=0.0` for greedy argmax and `temperature > 0` for sampling.

## Evaluate: success rate over multiple episodes

Run the model for `EVAL_EPISODES` full episodes and report how often it reaches the goal. The KV-cache is reset at each episode boundary.

In [4]:
def model_step(inputs_per_env, outputs, cache):
    """Feed the last transition to the model and return the next inputs + updated cache."""
    obs = outputs[0]
    batch = [[{
        "action": inputs_per_env[0]["action"],
        "observation": obs["observation"],
        "reward": obs["reward"],
        "done": obs["done"],
    }]]
    with torch.no_grad():
        predictions, _, cache = model(batch, cache=cache, use_cache=True)
    action = model.get_action(predictions, temperature=0.0, num_actions=MAX_ACTIONS)
    return [{"action": action.squeeze().cpu()}], cache


episodes_done = 0
successes = 0
inputs_per_env = env.sample_random_inputs()
cache = None

while episodes_done < EVAL_EPISODES:
    outputs = env.step(inputs_per_env)
    done_code = int(outputs[0]["done"])

    if done_code in (1, 2, 3, 4):  # episode ended
        reward = float(outputs[0]["reward"])
        if reward > 0:
            successes += 1
        episodes_done += 1
        cache = None  # reset KV-cache at episode boundary

    inputs_per_env, cache = model_step(inputs_per_env, outputs, cache)

env.close()
print(f"Success rate: {successes}/{EVAL_EPISODES} = {successes / EVAL_EPISODES:.0%}")

Success rate: 20/20 = 100%


## Replay animation

Capture a fresh episode and play it back frame-by-frame.

In [5]:
env = make_env(EnvConfig(
    "Procedural-FrozenLake-v1",
    name="proc_frozenlake",
    seed=0,
    num_envs=1,
    episodes_per_task=5,
    kwargs={"is_slippery": False, "fixed_map": FIXED_MAP, "step_penalty": -0.04, "render_mode": "rgb_array"},
))

inputs_per_env = env.sample_random_inputs()
cache = None
frames = []

with torch.no_grad():
    for _ in range(ANIMATION_STEPS):
        outputs = env.step(inputs_per_env)
        frames.extend(env.render())
        inputs_per_env, cache = model_step(inputs_per_env, outputs, cache)

env.close()
print(f"Collected {len(frames)} frames")

/home/user/Repos/mouse-core/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Collected 50 frames


In [6]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = animation.FuncAnimation(fig, update, frames=frames, interval=200, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())